In [2]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent 
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
duck = connect_to_postgres_via_duckdb()
create_clean_account_name_macro(duck)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'
✓ Created clean_account_name macro


# create deutshland table

In [ ]:
duck.sql(
    f"""
    create or replace table deutshland_consolidated as (
        select *, 'berlin' as city
        from read_csv('../{OUTPUT_DIR / "berlin_firms_consolidated.csv"}')
        union all 
        select *, 'dusseldorf' as city
        from read_csv('../{OUTPUT_DIR / "dusseldorf_firms_consolidated.csv"}')
        union all 
        select *, 'frankfurt' as city
        select *
        from read_csv('../{OUTPUT_DIR / "frankfurt_firms_consolidated.csv"}')
        union all 
        select *, 'hamburg' as city
        from read_csv('../{OUTPUT_DIR / "hamburg_firms_consolidated.csv"}')
        union all 
        select *, 'kiel' as city
        from read_csv('../{OUTPUT_DIR / "kiel_firms_consolidated.csv"}')
        union all 
        select *, 'koln' as city
        from read_csv('../{OUTPUT_DIR / "koln_firms_consolidated.csv"}')
        union all 
        select *, 'munchen' as city 
        from read_csv('../{OUTPUT_DIR / "munchen_firms_consolidated.csv"}')
        union all 
        select *, 'rostock' as city
        from read_csv('../{OUTPUT_DIR / "rostock_firms_consolidated.csv"}')
        union all 
        select *, 'stuttgart' as city
        from read_csv('../{OUTPUT_DIR / "stuttgart_firms_consolidated.csv"}')
        union all
        select *, 'viersen' as city
        from read_csv('../{OUTPUT_DIR / "viersen_firms_consolidated.csv"}')       
    )
    """
    )

In [5]:
duck.sql(
    """
    select * from deutshland_consolidated
    """
)

┌─────────────┬───────────────┬───────────────────────────┬───────────────────────────────────────────────────────────────────────┬──────────────────────────────┬───────────────────────────────────────────────┬──────────────────────────────────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                            eb_mother_firm                             │        name_medisoft         │                  eb_address                   │   medisoft_operating_firm_address    │
│    int64    │    varchar    │          boolean          │                                varchar                                │           varchar            │                    varchar                    │               varchar                │
├─────────────┼───────────────┼───────────────────────────┼───────────────────────────────────────────────────────────────────────┼──────────────────────────────┼───────────────────────────────────────────────┼──────────────────────────────────────┤


# create easybill customers table

In [6]:
duck.sql("create table easybill_customers as from read_csv('/Users/adrienblanquer/Downloads/Contacts-Export-30_01_2026-10_57_43.csv')")

# join deutshland conso with easybill customers 

## ajouter les domains dans deutshland_conso

## repartir d'ici -> pour les rows qui ne matchent pas avec le nom, faire le match avec le domain

In [8]:
duck.sql(
    """
    select 
        deutshland_consolidated.*,
        zoho.Id,
        zoho.Account_Name,
        zoho.Account_Name_cleaned,
        zoho.Employees_int as Employees
    from deutshland_consolidated
    left join (
        select
            Id,
            Account_Name,
            clean_account_name(Account_Name) as Account_Name_cleaned,
            Employees::int as Employees_int,
            row_number() over (
                partition by clean_account_name(Account_Name)
                order by Employees::int desc nulls last
            ) as rn
        from pg.zoho.Accounts
    ) as zoho
        on clean_account_name(deutshland_consolidated.eb_mother_firm) = zoho.Account_Name_cleaned
        and zoho.rn = 1
    """
)

┌─────────────┬───────────────┬───────────────────────────┬───────────────────────────────────────────────────────────────────┬────────────────────────────────────────────┬────────────────────────────────────────────────────────┬───────────────────────────────────────────────┬────────────────────┬─────────────────────────────────────────────────────┬───────────────────────────────────────────┬───────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                          eb_mother_firm                           │               name_medisoft                │                       eb_address                       │        medisoft_operating_firm_address        │         Id         │                    Account_Name                     │           Account_Name_cleaned            │ Employees │
│    int64    │    varchar    │          boolean          │                              varchar                              │                  varchar                   │          

In [188]:
duck.sql("select * from pg.zoho.Accounts where  lower(Account_Name) ilike 'EVA %'")

┌────────────────────┬────────────────────┬─────────┬─────────────────────────────────┬──────────┬─────────┬────────────────┬─────────────────────────────────────────┬───────────────┬──────────────┬───────────┬─────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┬────────────────┬──────────┬────────────────────┬────────────────────┬───────────────────────────┬───────────────────────────┬──────────┬───────────────┬───────────────────────────┬───────────────────┬─────────────────┬──────────────┬───────────────┬─────────────────────┬────────────────┬──────────────┬───────────────┬─────────────────┬──────────────────┬─────────────┬────────────────────┬────────────────┬──────────────┬─────────┬────────────────┬────────────────┬────────────────────────────┬───────────────────┬────────────────────────────┬─────────┬──────────────┬───────────────────────────┬──────────────────────────────┬────────────────────────────┬─────────────────────

In [179]:
duck.sql(
    """
    select
        *,
        clean_account_name(eb_mother_firm) as eb_mother_firm_cleaned 
    from deutshland_consolidated 

    """
)

┌─────────────┬───────────────┬───────────────────────────┬───────────────────────────────────────────────────────────────────────┬──────────────────────────────┬───────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────────────────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                            eb_mother_firm                             │        name_medisoft         │                  eb_address                   │   medisoft_operating_firm_address    │       eb_mother_firm_cleaned        │
│    int64    │    varchar    │          boolean          │                                varchar                                │           varchar            │                    varchar                    │               varchar                │               varchar               │
├─────────────┼───────────────┼───────────────────────────┼───────────────────────────────────────────────────────────────────────┼─────

In [ ]:
duck.sql(
    """
    select distinct on(d.id_easybill) id_easybill, id_medisoft, eb_mother_firm, name_medisoft, coalesce(split("E-Mail", '@')[2], split("Weitere E-Mails", '@')[2]) as domain
    from deutshland_consolidated as d
    join easybill_customers
        on d.id_easybill = easybill_customers.Kundennummer
    where Typ = 'Kunde'
    order by id_easybill, id_medisoft
    """
)

┌─────────────┬──────────────────────────────────────┬─────────────────────────────────────────────────────────────┬──────────────────────────────────────────┬──────────────────────────────────┐
│ id_easybill │             id_medisoft              │                       eb_mother_firm                        │              name_medisoft               │              domain              │
│    int64    │               varchar                │                           varchar                           │                 varchar                  │             varchar              │
├─────────────┼──────────────────────────────────────┼─────────────────────────────────────────────────────────────┼──────────────────────────────────────────┼──────────────────────────────────┤
│   100000003 │ NULL                                 │ Ates & Partner BAG GbR                                      │ NULL                                     │ praxisates.de                    │
│   100020000 │ NULL     

In [ ]:
duck.sql(
    """
    with contact_domains as (
        select *,
        
        regexp_extract(split(Email, '@')[2], 
                '(?:https?://)?(?:www\\.)?([a-zA-Z0-9\\-\\.]+\\.[a-zA-Z]+)', 
                1
            ) as domain
        from pg.zoho.Contacts
    ), zoho_accounts as (
        select accounts.Id as id_zoho, accounts.Account_Name as zoho_name, coalesce(accounts.Website, contacts.domain) as zoho_domain
        from pg.zoho.Accounts
        join contact_domains contacts
            on accounts.Id = contacts.Account_Name
    ), easybill_customers_domains as (
    select 
        coalesce(split("E-Mail", '@')[2], split("Weitere E-Mails", '@')[2]) as domain, 
        *
    from easybill_customers
    )
    select distinct on(Kundennummer) deutshland_consolidated.*
    from deutshland_consolidated
    join easybill_customers_domains
        on easybill_customers_domains.Kundennummer = deutshland_consolidated.id_easybill
    join zoho_accounts
        on easybill_customers_domains.domain = zoho_accounts.zoho_domain
    where Typ = 'Kunde' 
    
    """
)

┌─────────────┬───────────────┬───────────────────────────┬────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────┬─────────────────────────────────────────────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                 eb_mother_firm                 │                         name_medisoft                         │                    eb_address                    │         medisoft_operating_firm_address         │
│    int64    │    varchar    │          boolean          │                    varchar                     │                            varchar                            │                     varchar                      │                     varchar                     │
├─────────────┼───────────────┼───────────────────────────┼────────────────────────────────────────────────┼───────────────────────────────────────────────────────────────┼──────

In [150]:
duck.sql(
    """
    select coalesce(split(E_Mail, '@')[2], split(Website, '@')[2]) as domain
    from pg.zoho.Accounts 
    where Employees::int > 0 and Parent_Account is null
    order by Account_Name
    """
)

┌─────────────────────┐
│       domain        │
│       varchar       │
├─────────────────────┤
│ NULL                │
│ NULL                │
│ de.kline.com        │
│ mic-arc.de          │
│ NULL                │
│ NULL                │
│ NULL                │
│ NULL                │
│ NULL                │
│ 1000hands.de        │
│  ·                  │
│  ·                  │
│  ·                  │
│ NULL                │
│ NULL                │
│ NULL                │
│ oekohaus-rostock.de │
│ NULL                │
│ NULL                │
│ NULL                │
│ lasermed.de         │
│ NULL                │
│ NULL                │
├─────────────────────┤
│      6103 rows      │
│     (20 shown)      │
└─────────────────────┘

In [94]:
duck.sql(
    """
    select 
        Id, 
        Account_Name,
        regexp_extract(
            coalesce(Website, split(E_Mail, '@')[2]), 
            '(?:https?://)?(?:www\\.)?([a-zA-Z0-9\\-\\.]+\\.[a-zA-Z]+)', 
            1
        ) as domain
    from pg.zoho.Accounts  
    where domain is not null
        and Account_Name ilike '%worx%'
    """
)

┌─────────┬──────────────┬─────────┐
│   Id    │ Account_Name │ domain  │
│ varchar │   varchar    │ varchar │
├─────────┴──────────────┴─────────┤
│              0 rows              │
└──────────────────────────────────┘

In [68]:
duck.sql(
    """
    select * from deutshland_consolidated where id_easybill = '600740000'
    """
    )

┌─────────────┬──────────────────────────────────────┬───────────────────────────┬────────────────┬───────────────┬───────────────────────────────────────┬───────────────────────────────────────┐
│ id_easybill │             id_medisoft              │ should_have_medisoft_firm │ eb_mother_firm │ name_medisoft │              eb_address               │    medisoft_operating_firm_address    │
│    int64    │               varchar                │          boolean          │    varchar     │    varchar    │                varchar                │                varchar                │
├─────────────┼──────────────────────────────────────┼───────────────────────────┼────────────────┼───────────────┼───────────────────────────────────────┼───────────────────────────────────────┤
│   600740000 │ 871B8360-72A5-4800-8CF2-E27A49398133 │ true                      │ HY Studio GmbH │ HY Studio     │ Rosa-Luxemburg-Straße 20 10178 Berlin │ Rosa-Luxemburg-Straße 20 10178 Berlin │
└─────────────┴─────

In [71]:
duck.sql(
    """
    select * from easybill_customers where Typ = 'Kunde'
    """
)

┌────────────┬──────────────┬───────────────────┬───────────┬─────────┬────────────────────────┬─────────┬───────────────┬─────────────┬──────────────┬─────────────────────────────────────────────────────────────┬──────────────────────────┬──────────────┬───────────────────┬──────────────────────┬─────────────┬─────────────┬──────────┬───────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────────────────┬────────────────────┬───────────┬─────────┬──────────────┬─────────────────────────────────────┬────────────────────────────────────────────┬──────────┬─────────┬─────────┬────────────┬─────────────────────┬─────────────────┬────────────────┬──────────────┬──────────┬─────────────┬──────────────┬─────────────┬────────────────────────┬──────────┬──────────┬──────────────────────────────────────────────────────────────────────────┬────────────────┬──────────────────────┬───────────────────────┬────────────────────────┬────────────────────────────────────────┬

# zoho parent_account analysis

In [5]:
duck.sql(
    """
    select parent.id, parent.Account_Name as parent_name, child.id, child.Account_Name as child_name
    from pg.zoho.Accounts as child
    join pg.zoho.Accounts as parent
        on parent.id = child.Parent_Account
        and child.Parent_Account is not null
        and parent.Parent_Account is not null
    """
)

┌────────────────────┬───────────────────────────────────────────────────────────────────────┬────────────────────┬──────────────────────────────────────────────────┐
│         Id         │                              parent_name                              │         Id         │                    child_name                    │
│      varchar       │                                varchar                                │      varchar       │                     varchar                      │
├────────────────────┼───────────────────────────────────────────────────────────────────────┼────────────────────┼──────────────────────────────────────────────────┤
│ 386758000012226899 │ Botschaft für Kinder gGmbH                                            │ 386758000012226899 │ Botschaft für Kinder gGmbH                       │
│ 386758000012251329 │ Regio Kliniken GmbH                                                   │ 386758000012251329 │ Regio Kliniken GmbH                              

In [28]:
duck.sql("select distinct coalesce(Website, split(E_Mail, '@')[2]) as domain, Account_Name from pg.zoho.Accounts where domain is not null")

┌───────────────────────────────────┬────────────────────────────────────────────────────────────────────────────┐
│              domain               │                                Account_Name                                │
│              varchar              │                                  varchar                                   │
├───────────────────────────────────┼────────────────────────────────────────────────────────────────────────────┤
│ www.sana.de                       │ MVZ Pinneberg                                                              │
│ www.semcoglas.de                  │ Semcoglas Holding GmbH                                                     │
│ https://www.la-red.de             │ la red GmbH                                                                │
│ https://www.keoz.com/             │ KEOZ GmbH                                                                  │
│ kian-service.de                   │ KIAN Service GmbH                         

In [30]:
duck.sql(
    """
    select  Kundennummer, Firma, coalesce(split("E-Mail", '@')[2], split("Weitere E-Mails", '@')[2]) as domain, *
    from easybill_customers
    where domain is not null
    and domain like '%sana.de'
    """
)

┌──────────────┬─────────────────────────────────────────────────┬─────────┬────────────┬──────────────┬───────────────────┬───────────┬─────────┬────────────────────────┬─────────┬────────────┬─────────┬──────────────┬─────────────────────────────────────────────────┬───────────────────┬──────────────┬──────────┬────────────┬─────────────┬─────────────┬──────────┬───────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────────────┬───────────┬─────────┬──────────────┬────────────────────────────┬─────────────────┬──────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬─────────────

# a

In [8]:
duck.sql(
    """
    select distinct id_medisoft, eb_mother_firm, name_medisoft, id_zoho, Account_Name
    from deutshland_consolidated
    left join pg.medisoft.table_firms_zoho
        on deutshland_consolidated.id_medisoft = pg.medisoft.table_firms_zoho.rec_id
    left join pg.zoho.Accounts
        on pg.medisoft.table_firms_zoho.id_zoho = pg.zoho.Accounts.id
    where id_medisoft is not null
    """
)

┌───────────────┬────────────────────────────────────────────────────────┬──────────────────────────────────────────────────┬────────────────────┬──────────────────────────────────────────────────────────────────────┐
│  id_medisoft  │                     eb_mother_firm                     │                  name_medisoft                   │      id_zoho       │                             Account_Name                             │
│    varchar    │                        varchar                         │                     varchar                      │      varchar       │                               varchar                                │
├───────────────┼────────────────────────────────────────────────────────┼──────────────────────────────────────────────────┼────────────────────┼──────────────────────────────────────────────────────────────────────┤
│ 00_8ZT00TSYD2 │ ProVeg e.V.                                            │ ProVeg e.V.                                      │ 38

In [64]:
duck.sql(
    """
    create or replace table zoho_eb_matched as (
    select distinct on(id_easybill) id_easybill, id_medisoft, eb_mother_firm, z.id, z.Account_Name
    from deutshland_consolidated
    join pg.zoho.Accounts as z
        on (clean_account_name(z.Account_Name) = clean_account_name(deutshland_consolidated.eb_mother_firm)
        or clean_account_name(deutshland_consolidated.eb_mother_firm) in clean_account_name(z.Account_Name))
    where z.Parent_Account is null and Datum_Vertragsbeginn is not null
)
    """
)

In [65]:
duck.sql("select id, Account_Name from pg.zoho.Accounts where Parent_Account is null and 'juit' in lower(Account_Name)")

┌────────────────────┬──────────────┐
│         Id         │ Account_Name │
│      varchar       │   varchar    │
├────────────────────┼──────────────┤
│ 386758000010024238 │ Juit GmbH    │
└────────────────────┴──────────────┘

In [76]:
duck.sql("from zoho_eb_matched where 'ejf' in lower(eb_mother_firm)")

┌─────────────┬───────────────┬──────────────────────┬────────────────────┬──────────────────────┐
│ id_easybill │  id_medisoft  │    eb_mother_firm    │         Id         │     Account_Name     │
│    int64    │    varchar    │       varchar        │      varchar       │       varchar        │
├─────────────┼───────────────┼──────────────────────┼────────────────────┼──────────────────────┤
│   105010000 │ 00_8MT00LFIAA │ EJF gemeinnützige AG │ 386758000011736174 │ EJF gemeinnützige AG │
└─────────────┴───────────────┴──────────────────────┴────────────────────┴──────────────────────┘

In [79]:
duck.sql(
    """
    select  distinct on(deutshland_consolidated.id_easybill) * 
    from deutshland_consolidated
    left join zoho_eb_matched
        on zoho_eb_matched.id_easybill = deutshland_consolidated.id_easybill
    where id is null
    --where  'ejf' in    lower(zoho_eb_matched.eb_mother_firm) 


    """
)

┌─────────────┬───────────────┬───────────────────────────┬─────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────┬──────────────────────────────────────────┬─────────────┬─────────────┬────────────────┬─────────┬──────────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                           eb_mother_firm                            │                                  name_medisoft                                   │                       eb_address                       │     medisoft_operating_firm_address      │ id_easybill │ id_medisoft │ eb_mother_firm │   Id    │ Account_Name │
│    int64    │    varchar    │          boolean          │                               varchar                               │                                     varchar                                      │                      